# GeoAI benchmark on a free Colab T4: compact-schema A/B, base vs LoRA, candidate models

Runs the GeoAI benchmark harness (`python -m core.geoai.eval.benchmark`) on a GPU through
`core/geoai/eval/remote/run_remote_benchmark.sh`, the same script used on AWS
(see `core/geoai/eval/remote/README.md`), so the settings, skip/resume logic and run metadata match.

Order: 1 GPU check -> 2 code -> 3 install (CUDA llama-cpp-python) -> 4 jobs (compact-ab, lora, models, full-test)
-> 5 leaderboard + desktop-latency estimate -> 6 zip + download.

* Runtime: GPU (Runtime -> Change runtime type -> T4). No secrets: models and code are public.
* **After fine-tuning:** run the fine-tune notebook ([GeoAI_finetune.ipynb](https://colab.research.google.com/github/utkarsh-1912/geocore/blob/main/python-backend/core/geoai/finetune/colab/GeoAI_finetune.ipynb),
  `core/geoai/finetune/colab/`) first. Its last section runs the base-vs-adapter benchmark in the same
  runtime. Running this notebook in that same runtime also works: the adapter is still at `/content/geoai-ft/export/`.
* GPU latency is not desktop latency. Step 5 adds **estimated** desktop latency from token counts
  (`core/geoai/eval/data/desktop_calibration.json`).
* Times are estimates for a T4, not measurements.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Get the GeoCore code
Same location as the fine-tune notebook. The code must include the benchmark changes (`--label-suffix`, token counts,
`estimate_desktop_latency`). Commit and push them first, or upload a zip of `python-backend/` and unzip it to `/content/geocore/python-backend`.

In [ ]:
import os
if not os.path.exists('/content/geocore'):
    !git clone --depth 1 https://github.com/utkarsh-1912/geocore.git /content/geocore
%cd /content/geocore/python-backend
!git log -1 --format='%h %s' 2>/dev/null || true

## 3. Install: prebuilt CUDA llama-cpp-python 0.3.35 (cu124 wheel, ~1.7 GB) + backend requirements (~3-6 min)
`NO_VENV=1` installs into this notebook's Python. The script checks that the build has GPU offload, then regenerates
the (git-ignored) eval data deterministically if it is missing.

In [ ]:
S = 'core/geoai/eval/remote/run_remote_benchmark.sh'
os.environ.update(NO_VENV='1', WORK_DIR='/content/geoai-bench', RUN_PREFIX='colab-t4')
!bash {S} setup

## 4a. Compact-schema A/B (~5-10 min)
`qwen3-1.7b`, `GEOAI_COMPACT_SCHEMAS=0` vs `1`, the same stratified 40 `eval_test` examples, decision mode,
`n_ctx` 4096, 5 tools, native chat format (the cpu-test40 settings), all layers on the GPU. Each config runs in its own process.

In [ ]:
!bash {S} compact-ab

## 4b. Base vs fine-tuned LoRA on the full `eval_test` split (~20-40 min)
Point `LORA` at the exported GGUF adapter; its sidecar `geoai-lora.gguf.json` must sit next to it. After the fine-tune
notebook in this runtime it is `/content/geoai-ft/export/geoai-lora.gguf`; otherwise upload both files.
The script writes `remote/comparisons/go_no_go.json` (rule in `core/geoai/finetune/README.md`).
The p50-latency criterion there is GPU latency; check the desktop estimate in step 5.

In [ ]:
LORA = '/content/geoai-ft/export/geoai-lora.gguf'
!bash {S} --lora {LORA} lora

## 4c. Optional: all candidate models on the 40 examples (~30-70 min incl. ~29 GB of downloads)
Narrow it with `MODELS`, e.g. `os.environ['MODELS'] = 'qwen3-1.7b,qwen3-4b-2507'`.

In [ ]:
# !bash {S} models

## 4d. Optional: full `eval_test` split, qwen3-1.7b + heuristic (~10-20 min)

In [ ]:
# !bash {S} full-test

## 5. Leaderboards and estimated desktop latency
Quality comes from the GPU run. `est p50/p95` = tokens / desktop throughput measured on the reference laptop. These
are **estimates**, not measurements.

In [ ]:
import glob
for d in sorted(glob.glob('core/geoai/eval/results/benchmark/colab-t4-*')):
    run = os.path.basename(d)
    !python -m core.geoai.eval.benchmark leaderboard --run-id {run}
    !python -m core.geoai.eval.estimate_desktop_latency --run-id {run} --no-write
    for c in glob.glob(f'{d}/remote/comparisons/*.json'):
        print(c)

## 6. Zip and download
Unpack the archive inside `python-backend/` on the desktop; the files land in `core/geoai/eval/results/benchmark/`.

In [ ]:
!cd /content/geocore/python-backend && zip -qr /content/geoai-benchmark-results-colab.zip core/geoai/eval/results/benchmark/colab-t4-* && ls -la /content/geoai-benchmark-results-colab.zip
from google.colab import files
files.download('/content/geoai-benchmark-results-colab.zip')